# 🛡️ Glu-Stock: 00a_MODEL_RETRAINING_RF
**Phase**: Dynamic Cross-Sectional Intelligence (RF Brain)

This notebook trains the Random Forest 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the model stays relevant to the most liquid assets.

In [7]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib lxml html5lib python-dotenv



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Web Fetchers)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, firestore
from datetime import datetime

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as url:
            data = json.loads(url.read().decode())
            return [f"{t}.JK" for t in data]
    except:
        pass
    return fallback

def get_full_idx_universe():
    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")
    fallback = get_dynamic_lq45() 
    
    # 1. Try Official IDX API
    try:
        import urllib.request
        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            if 'data' in data:
                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]
                if tickers:
                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")
                    return list(set(tickers)) 
    except Exception as e:
        print(f"⚠️ Official IDX API failed. Trying Github Proxy...")
        
    # 2. Try Github Alternative
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]
            if tickers:
                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")
                return list(set(tickers))
    except Exception as e:
        print("⚠️ Full fetch failed. Falling back to LQ45.")
        
    return fallback
\n

In [9]:
# 🧠 SECTION 3: CORE LOGIC (Panel Training Pipeline)
from sklearn.ensemble import RandomForestClassifier

def build_panel_data(universe, period="5y"):
    all_X, all_y = [], []
    print(f"📉 Fetching {period} of data for {len(universe)} tickers...")
    for ticker in universe:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > 100:
            X = df[['Close']].pct_change().dropna().values.reshape(-1, 1)
            y = (df['Close'].shift(-1) > df['Close']).iloc[:-1].values.astype(int)
            all_X.append(X[:len(y)])
            all_y.append(y)
    print(f"✅ Successfully aggregated market data.")
    if len(all_X) == 0:
        print('❌ ERROR: NO DATA FETCHED! Check your internet connection (DNS issue with Yahoo) or yfinance version.')
        return None, None
    return np.vstack(all_X), np.concatenate(all_y)

def train_rf(X, y):
    print(f"🧠 Training Super RF Brain on {len(y)} samples...")
    model = RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=5)
    model.fit(X, y)
    return model

In [10]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = "/kaggle/working/"
    
    universe = get_full_idx_universe()
    
    # 1. Build Panel & Train RF
    X_train, y_train = build_panel_data(universe)
    if X_train is None:
        print('⚠️ Retraining aborted due to data fetch failure.')
        return
    rf_model = train_rf(X_train, y_train)
    accuracy = rf_model.score(X_train, y_train)
    
    brain_data = {"model": rf_model, "features": ["Close"], "accuracy": accuracy, "trained_at": datetime.now().isoformat()}
    joblib.dump(brain_data, os.path.join(output_dir, "glu_brain_v1.joblib"))
    
    fb.log_event("RETRAINING_RF", f"Completed RF panel update on {len(universe)} dynamic LQ45 tickers (Acc: {accuracy:.2f}).")
    print(f"✅ RF Model updated successfully in WORKING directory with {len(y_train)} experiences.")

run_retrain()

🌐 Fetching latest LQ45 constituents...
📉 Fetching 5y of data for 45 tickers...



1 Failed download:
['ACES.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ADRO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AKRA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMMN.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMRT.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ANTM.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ARTO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ASII.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBCA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBNI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBRI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBTN.JK']: TypeError("'NoneType' object is 

✅ Successfully aggregated market data.


ValueError: need at least one array to concatenate